# Uncertainty quantification for reactive transport modellers
> ### _"All models are wrong, but some are useful."_
> -- George Box

This is a background notebook in the **part0** series. The part0 notebooks are standalone: you can read them in any order and you do not need to run anything upstream first.

If you build reactive transport models (RTMs) but have never met PEST, this notebook is the one to start with. It introduces the handful of ideas that the rest of this curriculum leans on the whole way through: treating parameters as probability distributions, working with ensembles of model runs instead of a single calibrated model, the difference between *prior* and *posterior*, and Bayes' theorem in pictures. We deliberately keep the theory compressed -- the goal is intuition, not a derivation.

The companion part0 notebooks fill in the rest of the background:

- ["intro to DIZON"](../part0_01_intro_to_dizon/intro_to_dizon.ipynb) -- the field experiment, the chemistry, and the decision question this whole series is built around.
- ["intro to mf6rtm"](../part0_02_intro_to_mf6rtm/intro_to_mf6rtm.ipynb) -- the MODFLOW 6 + PHREEQC API, demonstrated on a tiny pyrite column.
- ["intro to PEST and IES"](../part0_04_intro_to_pest_and_ies/intro_to_pest_and_ies.ipynb) -- the model-as-black-box contract and how `PESTPP-IES` works.
- ["intro to DSI"](../part0_05_intro_to_dsi/intro_to_dsi.ipynb) -- emulation in observation space, the engine of the part1 workflow.

We are not going to re-derive uncertainty theory here. The GMDSI groundwater-modelling tutorials do that thoroughly and well, and this curriculum is a deliberate sibling of theirs. For depth on any concept below, see the **part0** notebooks in the [GMDSI_notebooks repository](https://github.com/gmdsi/GMDSI_notebooks) -- in particular their `intro_to_bayes` and `intro_to_pyemu` notebooks. The didactic Bayes example further down borrows from theirs, compressed hard.

### Admin

This notebook is self-contained. It does not run the reactive transport model, does not call PEST, and does not read any prepared dataset -- everything below is a small `numpy`/`matplotlib` illustration you can run in seconds. Run the cells with `shift+enter`.

All we need are the standard scientific-Python libraries:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.size'] = 10
rng = np.random.default_rng(2025)

## 1. A parameter is not a number

When you build an RTM you have to put numbers into it: a hydraulic conductivity, a porosity, a pyrite abundance, a reaction rate. The temptation is to treat each of these as *the* value -- the one you looked up, fitted, or were handed. But none of them is known exactly. The aquifer is in the ground; you cannot measure pyrite abundance everywhere, and the rate constant you borrowed from a column experiment may not be the rate constant of *this* aquifer at *this* temperature.

So the honest representation of a parameter is not a number but a **probability distribution**: a statement of which values are plausible and how plausible each is. A best guess plus an admission of how wrong that guess might be.

Take pyrite abundance, the signature uncertainty of the DIZON case. Suppose our best estimate is around 5 mmol per litre of bulk volume, but we would not be surprised by half or double that. We can write that down as a log-normal distribution:

In [ ]:
# pyrite abundance prior (mmol/L bulk), expressed in log10 space
log_mean = np.log10(5.0)   # best guess ~ 5 mmol/L
log_std = 0.3              # ~ a factor of 2 either side

samples = 10 ** rng.normal(log_mean, log_std, size=5000)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(samples, bins=60, color='0.6', edgecolor='none')
ax.axvline(5.0, color='b', lw=2, label='best guess (5 mmol/L)')
ax.set_xlabel('pyrite abundance (mmol/L bulk)')
ax.set_ylabel('count')
ax.set_title('a parameter as a probability distribution')
ax.legend()
fig.tight_layout()

The single "best guess" is just the centre of a spread. Everything in this curriculum follows from taking that spread seriously instead of discarding it.

## 2. Realisations and ensembles

A real model has many uncertain parameters at once, not one. If we draw a value for *each* parameter from its distribution, we get one complete, plausible parameter set -- one possible version of the aquifer. Each such parameter set is called a **"realisation"**. The suite of realisations is called an **"ensemble"**.

Here is a toy two-parameter ensemble -- pyrite abundance and hydraulic conductivity, each drawn from its own prior. Each dot is one realisation; the cloud of dots is the ensemble:

In [ ]:
n_real = 201   # the curriculum draws and keeps 201 realisations

pyrite = 10 ** rng.normal(np.log10(5.0), 0.3, size=n_real)   # mmol/L bulk
k = 10 ** rng.normal(np.log10(10.0), 0.4, size=n_real)       # m/d

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(pyrite, k, s=18, color='0.4', alpha=0.7)
ax.set_xlabel('pyrite abundance (mmol/L bulk)')
ax.set_ylabel('hydraulic conductivity (m/d)')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_title(f'an ensemble of {n_real} realisations')
fig.tight_layout()

We use **201** realisations throughout this curriculum -- enough to characterise the spread of outcomes without paying for runs we do not need. Hold that number; it recurs everywhere.

The point of an ensemble is what happens when you *run the model* for every realisation. Instead of one prediction you get 201 predictions -- a distribution of forecasts that carries the parameter uncertainty straight through the model to the answer you actually care about.

## 3. Prior and posterior

The ensemble we just drew encodes what we believe **before** looking at any monitoring data. That is the **prior**. It comes from soft knowledge: literature values, the geology, expert judgment, the bounds you are willing to defend.

Then data arrive. In the DIZON case, monitoring wells record breakthrough of SO₄, O₂, NO₃, pH and temperature during the **history period** (day 0–252). Some realisations reproduce those observations well; others do not. The data let us *re-weight* the ensemble -- keeping the realisations consistent with what was measured, discounting the ones that are not. What remains is the **posterior**: what we believe **after** conditioning on the data.

A one-parameter cartoon makes the relationship concrete. The prior is broad. The data favour a particular range. The posterior is the prior pulled toward the data -- narrower, but never narrower than the data alone can support:

In [ ]:
x = np.linspace(0, 20, 400)

prior = stats.norm(loc=10, scale=4).pdf(x)        # broad prior belief
likelihood = stats.norm(loc=14, scale=2).pdf(x)   # what the data favour

# for Gaussians the posterior is again Gaussian; combine precisions (1/variance)
post_var = 1.0 / (1 / 4**2 + 1 / 2**2)
post_mean = post_var * (10 / 4**2 + 14 / 2**2)
posterior = stats.norm(loc=post_mean, scale=np.sqrt(post_var)).pdf(x)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, prior, color='0.5', lw=2, label='prior (what we knew)')
ax.plot(x, likelihood, 'g--', lw=2, label='likelihood (what the data say)')
ax.plot(x, posterior, 'b', lw=2.5, label='posterior (what we know now)')
ax.set_xlabel('parameter value')
ax.set_ylabel('probability density')
ax.set_title('prior + data = posterior')
ax.legend()
fig.tight_layout()

Conditioning is not about finding the single "calibrated" value at the peak of that blue curve. It is about reshaping the *whole distribution*. The width of the posterior is the part that matters for decisions, and it is the part a single calibrated number throws away.

## 4. Bayes' theorem, in pictures

The machinery behind "prior + data = posterior" is Bayes' theorem. Written out:

$$\underbrace{P(\boldsymbol{\theta}|\textbf{d})}_{\substack{\text{posterior} \\ \text{what we know now}}} \propto \underbrace{\mathcal{L}(\boldsymbol{\theta}|\textbf{d})}_{\substack{\text{likelihood} \\ \text{what we learned}}} \;\; \underbrace{P(\boldsymbol{\theta})}_{\substack{\text{prior} \\ \text{what we knew}}}$$

where $\boldsymbol{\theta}$ are the parameters and $\textbf{d}$ are the data. In words: the posterior is proportional to the prior multiplied by how well each parameter set explains the data. That is the whole idea. (For the derivation from conditional probabilities, see the `intro_to_bayes` notebook in the [GMDSI_notebooks repo](https://github.com/gmdsi/GMDSI_notebooks).)

Let's *do* it once, by hand, in one dimension -- no PEST, just `numpy`. Imagine we are trying to pin down a single pyrite reaction-rate parameter. We start with a broad prior. Then we make one measurement: a downstream SO₄ concentration, which depends on the rate. The likelihood scores each candidate rate by how close its predicted SO₄ lands to what we measured.

First, the prior over the rate parameter, on a grid:

In [ ]:
rate = np.linspace(0, 10, 500)          # candidate rate values
drate = rate[1] - rate[0]               # grid spacing, for normalising
prior = stats.norm(loc=4.0, scale=2.5).pdf(rate)
prior /= (prior.sum() * drate)          # normalise to a density

Now a (deliberately simple) forward model: predicted SO₄ rises with the rate. We measured SO₄ = 240 mg/L, with measurement noise of about 15 mg/L. The likelihood is how probable that measurement is for each candidate rate:

In [ ]:
def forward(r):
    """toy forward model: SO4 (mg/L) produced for a given pyrite rate."""
    return 40.0 * r + 30.0

measured_so4 = 240.0
noise_std = 15.0

predicted = forward(rate)
likelihood = stats.norm(loc=measured_so4, scale=noise_std).pdf(predicted)

Multiply prior by likelihood, normalise, and we have the posterior -- Bayes' theorem in three lines:

In [ ]:
posterior = prior * likelihood
posterior /= (posterior.sum() * drate)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(rate, prior, color='0.5', lw=2, label='prior')
ax.plot(rate, likelihood / (likelihood.sum() * drate), 'g--', lw=2, label='likelihood (scaled)')
ax.plot(rate, posterior, 'b', lw=2.5, label='posterior')
ax.set_xlabel('pyrite reaction-rate parameter')
ax.set_ylabel('probability density')
ax.set_title('Bayes by hand: one parameter, one measurement')
ax.legend()
fig.tight_layout()

One measurement already moves us. The posterior is narrower than the prior and shifted toward the rates that explain the observed SO₄. More (and more informative) data would tighten it further -- but never to a spike, because the measurement itself is noisy. This grid-multiply is exactly what `PESTPP-IES` does at scale, for hundreds of parameters at once, without ever enumerating a grid. We meet it in the ["intro to PEST and IES"](../part0_04_intro_to_pest_and_ies/intro_to_pest_and_ies.ipynb) notebook.

## 5. Why a single calibrated model fails the decision

The traditional workflow is: adjust parameters until the model fits the historical data, declare that one parameter set "calibrated", and use it to predict the future. It is seductive because it gives you a single tidy answer. For a *decision*, it is dangerous.

The DIZON decision question is a design question: **how much sulfate will the supplied water carry during the supply period (day 308–728), and how sure are we?** Treatment cost scales with concentration, so the operator sizes treatment capacity to the high end of what is plausible — the P95 of peak SO₄, not its single best estimate. A single calibrated model gives one peak-SO₄ number and no spread, so it cannot tell you where that P95 sits. And the calibrated number is fragile: many different parameter sets fit the history about equally well, yet predict the future quite differently.

Let's see that happen. Below, every realisation in an ensemble is filtered to those that fit a historical observation acceptably -- the "behavioural" set, all roughly equally calibrated. Then we look at what those same realisations forecast for peak SO₄ at the supply well (`wellopt`):

In [ ]:
# toy ensemble: two parameters drive both the history fit and the forecast
n = 2000
p1 = rng.normal(0, 1, n)   # e.g. a transport parameter
p2 = rng.normal(0, 1, n)   # e.g. a reaction parameter

# historical observation depends mostly on p1; the forecast depends on both.
# coefficients are tuned so the behavioural forecast echoes the real curriculum
# numbers: median ~82, 5-95% ~72-94 mg/L (see the history-matching notebooks).
hist_sim = 100 + 20 * p1 + 4 * p2 + rng.normal(0, 2, n)
forecast = 83 + 1.5 * p1 + 7.5 * p2          # peak SO4 (mg/L) over the supply period

hist_obs = 100.0   # the single measured historical value
behavioural = np.abs(hist_sim - hist_obs) < 5.0   # 'good enough' calibration

All the behavioural realisations fit the past. Now compare their forecasts to the full prior:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(hist_sim, forecast, s=8, color='0.7', label='all realisations')
axes[0].scatter(hist_sim[behavioural], forecast[behavioural], s=10, color='b',
                label='fits the history')
axes[0].axvline(hist_obs, color='g', lw=2, label='measured history')
axes[0].set_xlabel('simulated historical observation')
axes[0].set_ylabel('forecast: peak SO$_4$ (mg/L)')
axes[0].legend(fontsize=8)

f = forecast[behavioural]
axes[1].hist(f, bins=30, color='b', alpha=0.6)
axes[1].axvline(np.median(f), color='k', lw=2, label=f'median {np.median(f):.0f}')
axes[1].axvline(np.percentile(f, 95), color='r', lw=2,
                label=f'P95 {np.percentile(f, 95):.0f} (design target)')
axes[1].set_xlabel('forecast: peak SO$_4$ (mg/L)')
axes[1].set_ylabel('count (behavioural realisations)')
axes[1].set_title('forecasts among equally-calibrated models')
axes[1].legend(fontsize=8)
fig.tight_layout()

Look at the left panel: a tight vertical band of models all match the measured history, yet they spread across a wide range of forecasts. Pick any one of them, call it "the calibrated model", and you have picked one point out of that spread -- with no way to know whether it sits near the middle or out in the tail you have to design for.

The right panel is the answer the decision actually needs: the *distribution* of plausible, history-consistent forecasts. Its median is the central expectation; its P95 is the high end the operator sizes treatment to. A single calibrated model gives you neither -- it has no spread to read a P95 off.

In [ ]:
f = forecast[behavioural]
print(f'peak SO4 forecast (mg/L) | fits the history:')
print(f'  median {np.median(f):.0f}, 5-95% {np.percentile(f, 5):.0f}-{np.percentile(f, 95):.0f}, P95 {np.percentile(f, 95):.0f}')

# a risk statement is a lens on the same distribution, not a different analysis:
# if your supply contract had a 90 mg/L trigger, the chance of tripping it is
p_trigger = (f > 90).mean()
print(f'  P(peak SO4 > 90 mg/L) = {p_trigger:.2f}')

That distribution -- a median and a P95 -- is the currency of every payoff figure in this curriculum. History matching is judged on what it does to the forecast distribution (prior vs posterior); data worth is judged on whether new data would tighten it; optimization is judged on the P95 it leaves the operator to design around. None of that is reachable from a single calibrated run. A contractual trigger like the 90 mg/L line above is just one lens on the same distribution; the EU drinking-water standard of 250 mg/L is comfortably met here, so it is cost, not compliance, that drives this decision.

## 6. The catch: RTM runs are expensive

Everything above asks you to run the model *many times* -- hundreds of realisations for the prior, and then many more during conditioning as `PESTPP-IES` iterates. For a fast model that is no problem. Reactive transport is not a fast model.

The DIZON model couples MODFLOW 6 groundwater flow and transport to PHREEQC geochemistry, solving the reaction network in every cell at every time step. On a MacBook, **one forward run takes about 6 minutes**. That single number reshapes the whole strategy:

In [ ]:
minutes_per_run = 6
n_real = 201

prior_mc_hours = minutes_per_run * n_real / 60
print(f'one prior Monte Carlo ({n_real} runs): {prior_mc_hours:.1f} hours')

# history matching with PESTPP-IES: several iterations, each an ensemble run
ies_iterations = 4
ies_runs = n_real * (ies_iterations + 1)
print(f'one IES history match (~{ies_iterations} iters): {ies_runs} runs, '
      f'{minutes_per_run * ies_runs / 60:.0f} hours')

# data worth: re-run the analysis for several observation subsets
dataworth_subsets = 6
print(f'data worth over {dataworth_subsets} obs subsets, the full-model way: '
      f'{minutes_per_run * ies_runs * dataworth_subsets / 60 / 24:.1f} days')

A single prior Monte Carlo is an afternoon. A full history match is most of a day. And the moment you want to ask a *second* question -- would different data have helped? what is the optimal pumping schedule? -- you are multiplying days by the number of variants you want to explore. For RTM, the brute-force ensemble workflow that works fine for a fast groundwater model is simply impractical. **That cost is the thesis of this curriculum.**

The way out is **emulation**. We pay the full-model cost *once* -- a single prior Monte Carlo of 201 runs -- and use those runs to train a fast surrogate that mimics the model's outputs. The surrogate this series uses is **DSI** (Data Space Inversion), which emulates directly in *observation space*: it learns the joint distribution of the model's outputs without ever re-running the model. Once trained, it conditions on data and answers decision questions in **seconds**, not days.

So the part1 workflow is emulation-first by necessity:

1. Run the prior Monte Carlo **once** (the expensive part -- about 20 hours of full-model runs, paid up front).
2. Train DSI on those runs and -- always -- **check its fidelity** against held-out realisations before trusting it.
3. Condition, score risk, evaluate data worth, and optimize, all on the fast emulator.

To keep the notebooks runnable, the expensive full-model results are pre-baked and tracked in the repository, so you load them rather than wait ~6 minutes per run. We say what was run and roughly what it cost each time.

## Recap

- A parameter is a **probability distribution**, not a single number.
- One draw from every parameter's distribution is a **realisation**; the suite is an **ensemble** (we use **201**).
- The **prior** is what we believe before data; the **posterior** is what we believe after conditioning. Bayes' theorem is the bridge: posterior $\propto$ likelihood $\times$ prior.
- A single **calibrated** model cannot answer a probabilistic decision question -- many models fit the history yet forecast differently. The decision needs the *spread*, summarised as the forecast distribution -- a median and a P95 the operator designs treatment around.
- RTM runs cost **~6 min each**, so brute-force ensembles are impractical. The series is **emulation-first**: pay the full-model cost once, then emulate (DSI) for everything else.

**Where to next:** if PEST is still a black box, read ["intro to PEST and IES"](../part0_04_intro_to_pest_and_ies/intro_to_pest_and_ies.ipynb); for the emulator that makes all of this affordable, read ["intro to DSI"](../part0_05_intro_to_dsi/intro_to_dsi.ipynb). For deeper treatment of any concept above, the part0 notebooks in the [GMDSI_notebooks repository](https://github.com/gmdsi/GMDSI_notebooks) are the canonical reference.